In [15]:
import numpy as np
import pandas as pd

from typing import Literal, Tuple, Dict, Optional
from autogluon.tabular import TabularDataset, TabularPredictor
from scipy.optimize import curve_fit
from scipy.stats import linregress
from pathlib import Path

In [ ]:
EPS = float(np.finfo(float).eps)

In [17]:
def __load_dfs(type: Literal["train", "test"]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    log_df = __load_log_df(type)
    
    flc_dfs = [__load_flc_df(type, split) for split in sorted(log_df["split"].unique())]
    flc_df = pd.concat(flc_dfs)

    return (log_df, flc_df)

def __load_log_df(type: Literal["train", "test"]) -> pd.DataFrame:
    return pd.read_csv(f"../resources/kaggle/{type}_log.csv", index_col="object_id").drop(columns=["English Translation"])

def __load_flc_df(type: Literal["train", "test"], split: str) -> pd.DataFrame:
    return pd.read_csv(f"../resources/kaggle/{split}/{type}_full_lightcurves.csv")

In [18]:
train_log_df, train_flc_df = __load_dfs("train")
test_log_df, test_flc_df = __load_dfs("test")

In [ ]:
class FeatureExtractor:
    """
    Extracts physically motivated features for TDE classification from light curves
    and metadata, adhering to theoretical constraints (fallback rate, constant temperature).
    """
    def __init__(self, log_df: pd.DataFrame, flc_df: pd.DataFrame):
        self.__log_df = log_df
        self.__flc_df = flc_df
        
        self.__flc_df = self.__apply_extinction_correction()

    def extract(self) -> pd.DataFrame:
        return self.__log_df.join([
            self.__extract_mean_blue_color(),
            self.__extract_color_evolution(),
            self.__extract_power_law_decay(),
            self.__extract_peak_luminosity(),
            self.__extract_lc_smoothness(),
        ])
        
    def __apply_extinction_correction(self) -> pd.DataFrame:
        """
        Corrects Flux for Galactic Extinction using E(B-V).
        Formula: F_corr = F_obs * 10^(0.4 * A_lambda)
        Approximation: A_lambda approx R_V * E(B-V) with R_V ~ 3.1
        """
        # Merge EBV from log_df into flc_df for row-wise calculation
        df = self.__flc_df.merge(self.__log_df[["EBV"]], left_on="object_id", right_index=True, how="left")
        
        # Apply correction formula
        df["Flux_corr"] = df["Flux"] * np.pow(10, 0.4 * 3.1 * df["EBV"])
        
        # Return the dataframe with the new column, dropping the merged EBV to keep it clean
        return df.drop(columns=["EBV"])
    
    def __extract_mean_blue_color(self) -> pd.DataFrame:
        """
        Derives mean 'u - g' color proxy.
        Formula: -2.5 * log10( Mean_Flux_u / Mean_Flux_g )
        Theory: TDEs are hot (Blue), Flux_u should be high relative to Flux_g.
        """
        return (
            self.__flc_df
            .groupby(["object_id", "Filter"])["Flux_corr"]
            .mean()
            .unstack()
            .pipe(lambda df: (df["u"] + EPS) / (df["g"] + EPS))
            .pipe(lambda ratio: -2.5 * np.log10(ratio.clip(lower=EPS)))
            .to_frame("mean_blue_color")
        )
    
    def __extract_color_evolution(self) -> pd.DataFrame:
        """
        Derives the rate of color change.
        Formula: Slope of (Flux_g / Flux_r) over time.
        Theory: TDEs have constant Temperature (slope ~ 0). SNe cool/redden (slope < 0).
        """
        # Filter for relevant bands 'g' and 'r'
        flc_df = self.__flc_df[self.__flc_df["Filter"].isin(["g", "r"])].copy()
        
        # Pivot to get time-aligned fluxes: Index=[object_id, Time], Columns=[g, r]
        # We take mean in case of multiple exposures at exact same MJD
        pivot_flc_df = flc_df.pivot_table(index=["object_id", "Time (MJD)"], columns="Filter", values="Flux_corr", aggfunc="mean").reset_index()
        
        # Calculate Color Ratio (g/r) at each time step
        pivot_flc_df["color_ratio"] = (pivot_flc_df["g"] + EPS) / (pivot_flc_df["r"] + EPS)
        
        # Remove noise (negative fluxes or NaNs)
        # pivot_flc_df = pivot_flc_df[pivot_flc_df['color_ratio'] > 0].dropna()

        slope_results = {}
        
        for obj_id, group in pivot_flc_df.groupby('object_id'):
            if len(group) < 3:
                slope_results[obj_id] = 0.0 # Insufficient data
                continue
                
            # Linear Regression: Time vs Color Ratio
            # We want to see if the ratio changes systematically over time
            slope, _, _, _, _ = linregress(group['Time (MJD)'], group['color_ratio'])
            
            # If slope is NaN (flat line or error), set to 0
            slope_results[obj_id] = 0.0 if np.isnan(slope) else slope

        df = pd.DataFrame.from_dict(slope_results, orient='index', columns=['color_evolution'])
        return df
    
    def __extract_power_law_decay(self) -> pd.DataFrame:
        """
        Derives the power-law decay index (alpha).
        Formula: log(Flux) = C - alpha * log(t - t_peak)
        Theory: TDEs follow t^-5/3 (alpha ~ 1.67).
        """
        decay_results = {}
        smoothness_results = {} # Calculating smoothness here as it uses the fit residuals
        
        # Group by object
        for obj_id, group in self.__flc_df.groupby('object_id'):
            # Use 'g' band for decay fit (usually high SNR for TDEs)
            # If 'g' is empty, use all bands aggregated
            g_band = group[group['Filter'] == 'g']
            fit_data = g_band if not g_band.empty else group
            
            if fit_data.empty:
                decay_results[obj_id] = 0.0
                smoothness_results[obj_id] = 1.0 # High error default
                continue

            # Identify Peak
            peak_idx = fit_data['Flux_corr'].idxmax()
            t_peak = fit_data.loc[peak_idx, 'Time (MJD)']
            
            # Select Post-Peak Data (e.g., > 10 days after peak to capture the tail)
            post_peak = fit_data[fit_data['Time (MJD)'] > (t_peak + 5)]
            
            # Filter for positive flux (cannot log negative flux)
            post_peak = post_peak[post_peak['Flux_corr'] > 0]
            
            if len(post_peak) < 5:
                decay_results[obj_id] = 0.0
                smoothness_results[obj_id] = 1.0
                continue
                
            # Prepare Log-Log data
            # time_delta = t - t_peak
            X = np.log10(post_peak['Time (MJD)'] - t_peak)
            Y = np.log10(post_peak['Flux_corr'])
            
            # Linear Fit
            slope, intercept, r_value, p_value, std_err = linregress(X, Y)
            
            # Feature 1: Decay Alpha (Slope should be negative, so we store -slope)
            # TDEs expect slope ~ -1.67, so stored value should be ~1.67
            decay_results[obj_id] = -slope if not np.isnan(slope) else 0.0
            
            # Feature 2: Smoothness (MSE of the fit)
            # Measures how much the lightcurve deviates from the theoretical power law
            y_pred = slope * X + intercept
            mse = np.mean((Y - y_pred) ** 2)
            smoothness_results[obj_id] = mse

        # Combine into DataFrame (handling the smoothness storage temporarily here)
        df_decay = pd.DataFrame.from_dict(decay_results, orient='index', columns=['power_law_decay'])
        self._temp_smoothness = smoothness_results # Store for the next function call
        
        return df_decay

    def __extract_peak_luminosity(self) -> pd.DataFrame:
        """
        Derives a proxy for Peak Absolute Magnitude/Luminosity.
        Formula: log10(Flux_peak) + 2 * log10(Redshift)
        Theory: L propto F * d^2. d propto z.
        """
        # Find max flux per object across all bands
        max_flux = self.__flc_df.groupby("object_id")["Flux_corr"].max()
        
        # Get Redshift
        z = self.__log_df["Z"]
        
        # Calculate Proxy
        # Avoid log(0) for z by adding small epsilon
        lum_proxy = np.log10(max_flux + 1e-6) + 2 * np.log10(z + 1e-6)
        
        df = pd.DataFrame(lum_proxy)
        df.columns = ["peak_luminosity"]
        return df

    def __extract_lc_smoothness(self) -> pd.DataFrame:
        """
        Returns the smoothness metric calculated during the decay fit.
        Formula: MSE of the log-log power law fit.
        Theory: Low MSE = Smooth (TDE). High MSE = Stochastic (AGN).
        """
        # Retrieve computed smoothness from the decay step to avoid re-computing
        if hasattr(self, '_temp_smoothness'):
            return pd.DataFrame.from_dict(self._temp_smoothness, orient='index', columns=['lc_smoothness'])
        else:
            # Fallback if called out of order (though extract() order ensures this exists)
            # Returns zeros structure
            return pd.DataFrame(0.0, index=self.__log_df.index, columns=['lc_smoothness'])

In [20]:
extractor = FeatureExtractor(train_log_df, train_flc_df)
extractor.extract()

,Z,Z_err,EBV,SpecType,split,target,mean_blue_color,color_evolution,power_law_decay,peak_luminosity,lc_smoothness
object_id,,,,,,,,,,,
Dornhoth_fervain_onodrim,3.0490,NaN,0.110,AGN,split_01,0,39.133899,0.0,0.000000,2.503477,1.000000
Dornhoth_galadh_ylf,0.4324,NaN,0.058,SN II,split_01,0,2.060340,0.0,0.495502,0.399664,0.090339
Elrim_melethril_thul,0.4673,NaN,0.577,AGN,split_01,0,4.881900,0.0,0.721199,0.875395,0.164438
Ithil_tobas_rodwen,0.6946,NaN,0.012,AGN,split_01,0,0.650503,0.0,0.036791,0.427015,0.234932
Mirion_adar_Druadan,0.4161,NaN,0.058,AGN,split_01,0,39.133899,0.0,0.000000,0.041460,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
tinnu_gellui_tathar,0.8898,NaN,0.042,AGN,split_20,0,39.133899,0.0,0.662363,0.758773,0.099940
uir_heleg_corf,0.9598,NaN,0.042,AGN,split_20,0,-0.906840,0.0,0.000000,0.883855,1.000000
uir_rhosc_law,0.1543,NaN,0.024,SN II,split_20,0,1.018638,0.0,0.000000,-0.887151,1.000000


In [21]:
# predictor = TabularPredictor(path = "../AutogluonModels", problem_type="binary", label="target").fit(train_df, presets = "medium")

# prediction_df = pd.DataFrame({
#     "object_id": test_df["object_id"],
#     "prediction": predictor.predict(test_df),
# })

# prediction_df

In [22]:
# Path("../output").mkdir(parents=True, exist_ok=True)

# prediction_df.to_csv("../output/submission.csv", index=False)